In [15]:
import pandas as pd
from pulp import *
from sklearn.cluster import KMeans
import plotly.graph_objs as go
import pandas as pd
import folium
from branca.colormap import linear

In [16]:
demanda = pd.read_csv('C:\\Users\\Lenovo\\OneDrive\\Documentos\\André Luis\\1. UFMG\\TCC\\TCC\\Resultados\\Utils\\df_demanda_quadrados.csv').drop(['Unnamed: 0'], axis=1)

quadrados = pd.read_csv('C:\\Users\\Lenovo\\OneDrive\\Documentos\\André Luis\\1. UFMG\\TCC\\TCC\\Resultados\\Utils\\df_clientes_quadrados.csv').drop(['Unnamed: 0', 'Order Postal Code', 'Latitude', 'Longitude'], axis=1).groupby(by='Quadrado').mean().reset_index()
quadrados['latCentroide'] = (quadrados['maxLat'] + quadrados['minLat'])/2
quadrados['longCentroide'] = (quadrados['maxLong'] + quadrados['minLong'])/2
#quadrados = quadrados.drop(['maxLat', 'minLat', 'maxLong', 'minLong'], axis=1)

demanda = demanda.merge(quadrados, how='left')

distancias = pd.read_csv('C:\\Users\\Lenovo\\OneDrive\\Documentos\\André Luis\\1. UFMG\\TCC\\TCC\\Resultados\\Utils\\df_distancias_cds.csv').drop(['Unnamed: 0'], axis=1)
distancias['quadrado'] = distancias['quadrado'].apply(int)

candidates = pd.read_excel('../dados.xlsx', sheet_name='candidates')[['Nome', 'Latitude', 'Longitude']]

In [ ]:
# Define o número de clusters
num_clusters = [1, 3 , 5, 7]

df_resultados = pd.DataFrame(columns=['num', 'cluster', 'candidate'])

for num in num_clusters:
    df_quadrados = quadrados
    df_quadrados['num'] = num

    # Define o modelo de clusterização K-means
    kmeans = KMeans(n_clusters=num)

    # Separa as colunas de latitude e longitude do dataframe de clientes
    X = df_quadrados[['latCentroide', 'longCentroide']]

    # Realiza a clusterização dos clientes
    kmeans.fit(X)

    # Adiciona o cluster atribuído a cada cliente no dataframe de clientes
    df_quadrados['cluster'] = kmeans.labels_

    # Itera sobre cada cluster
    for cluster in range(num):
        quadrados_cluster = df_quadrados[df_quadrados['cluster'] == cluster]
        distancia = distancias[distancias['quadrado'].isin(quadrados_cluster['Quadrado'])]
        df = distancia[['CD', 'Distance']].groupby(by='CD').mean().reset_index()
        candidate = df[df['Distance'] == df['Distance'].min()]['CD'].values[0]
        d = {'num': [num], 'cluster': [cluster], 'candidate': [candidate]}
        serie = pd.DataFrame(d)
        df_resultados = pd.concat([df_resultados, serie])

    if num == 1:
        # Salva o resultado
        df_resultado = df_quadrados.merge(df_resultados[['num', 'cluster', 'candidate']], left_on=['num', 'cluster'], right_on=['num', 'cluster'])
    else:
        df_resultado = pd.concat([df_resultado, df_quadrados.merge(df_resultados[['num', 'cluster', 'candidate']], left_on=['cluster', 'num'], right_on=['cluster', 'num'])])


In [23]:
df_resultado.to_csv('C:\\Users\\Lenovo\\OneDrive\\Documentos\\André Luis\\1. UFMG\\TCC\\TCC\\Resultados\\Utils\\quadrados_clusterizados_todos_os_clusters.csv')

### Mapa

In [98]:
for num in num_clusters:
    df_result = df_resultado[df_resultado['num'] == num]

    # Cria um mapa centralizado em uma localização inicial
    mapa = folium.Map(location=[-23.802923, -46.728143], zoom_start=12)

    # Define a escala de cores
    escala_cores = linear.YlOrRd_09.scale(0,num) 

    # Itera sobre cada linha do dataframe
    for index, row in df_result.iterrows():
        # Extrai as informações de cada linha
        quadrado = row['Quadrado']
        max_lat = row['maxLat']
        min_lat = row['minLat']
        max_long = row['maxLong']
        min_long = row['minLong']
        cluster = row['cluster']
        
        # Calcula a cor com base na quantidade normalizada
        cor = escala_cores(cluster)
        
        # Cria um retângulo no mapa representando o quadrado e colorea com a cor calculada
        folium.Rectangle(
            bounds=[(min_lat, min_long), (max_lat, max_long)],
            color='gray',
            weight=1,
            fill=True,
            fill_color=cor,
            fill_opacity = 0.8, 
            tooltip = row['Quadrado']
        ).add_to(mapa)

    for i in df_result['candidate'].unique():
        candidate = candidates[candidates['Nome'] == i]
        cluster = df_result[df_result['candidate'] == candidate['Nome'].values[0]].iloc[0]
        cor = escala_cores(cluster['cluster'])
        folium.Marker(
            location=[candidate['Latitude'], 
            candidate['Longitude']], 
            icon=folium.Icon(color='lightgray', 
            icon='industry', prefix='fa')).add_to(mapa)
        folium.CircleMarker(
            location=[candidate['Latitude'], candidate['Longitude']],
            color='black',
            weight=3,
            fill=True,
            fill_color=cor,
            fill_opacity = 0.8, 
            tooltip = row['Quadrado']
            ).add_to(mapa)

    escala_cores.caption = 'Cluster'
    escala_cores.add_to(mapa)

    sw = [df_result['minLat'].min(), df_result['minLong'].min()]
    ne = [df_result['maxLat'].max(), df_result['maxLong'].max()]
    bounds = [sw, ne]
    mapa.fit_bounds(bounds=bounds)

    # Salva o mapa
    folium.TileLayer('cartodbpositron').add_to(mapa)
    mapa.save("C:\\Users\\Lenovo\\OneDrive\\Documentos\\André Luis\\1. UFMG\\TCC\\TCC\\Resultados\\Maps\\" + str(num) + "clusters_cds.html")